In [ ]:
import jax
import jax.numpy as jnp
from jaxtyping import Array, Float
from tqdm import tqdm
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import scipy as sp
import numpy as np
from src import gp, acquisition, test_functions

jax.config.update("jax_enable_x64", True)

In [ ]:
MAX_ITERATIONS = 100
ACQUISITION_MULTISTARTS = 16
KERNEL = "matern52"
VERBOSE = True

# Target Function - Goldstein Price

In [ ]:
goldstein_price = test_functions.GoldsteinPrice()
x_grid = jnp.linspace(0, 1, 100)
y_grid = jnp.linspace(0, 1, 100)
x_meshgrid, y_meshgrid = jnp.meshgrid(x_grid, y_grid)
X_meshgrid = jnp.stack([x_meshgrid.flatten(), y_meshgrid.flatten()], axis=-1)

fig = go.Figure(
    data=[
        go.Surface(
            z=goldstein_price(X_meshgrid).reshape(x_meshgrid.shape),
            x=x_meshgrid,
            y=y_meshgrid,
        )
    ]
)
fig.show()

# Fit GP on initial points

In [ ]:
x0 = sp.stats.qmc.LatinHypercube(d=2, optimization="random-cd").random(n=4)
y0 = goldstein_price(x0)
model = gp.GaussianProcess.fit(x0, y0, kernel=KERNEL, max_iterations=MAX_ITERATIONS, verbose=VERBOSE)

In [ ]:
pred = model.predict(X_meshgrid)
fig = make_subplots(
    rows=1,
    cols=2,
    specs=[[{"type": "scene"}, {"type": "xy"}]],
    subplot_titles=("GP Prediction", "GP Uncertainty"),
)

# Left subplot: 3D GP mean + true surface + sampled points
fig.add_trace(
    go.Surface(
        z=pred.mean.reshape(x_meshgrid.shape),
        x=x_meshgrid,
        y=y_meshgrid,
        showscale=False,
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Surface(
        z=goldstein_price(X_meshgrid).reshape(x_meshgrid.shape),
        x=x_meshgrid,
        y=y_meshgrid,
        opacity=0.3,
        showscale=False,
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter3d(
        z=goldstein_price(x0),
        x=x0[:, 0],
        y=x0[:, 1],
        mode="markers",
        marker=dict(color="red", size=5),
    ),
    row=1,
    col=1,
)

# Right subplot: uncertainty contour + sampled points
fig.add_trace(
    go.Contour(
        z=jnp.sqrt(jnp.diag(pred.cov)).reshape(x_meshgrid.shape),
        x=x_grid,
        y=y_grid,
    ),
    row=1,
    col=2,
)
fig.add_trace(
    go.Scatter(
        x=x0[:, 0],
        y=x0[:, 1],
        mode="markers",
        marker=dict(color="red", size=10),
    ),
    row=1,
    col=2,
)

fig.update_layout(width=800, height=400)
fig.show()

# Expected Improvement

In [ ]:
bfgs_acquisition = acquisition.BFGS()
ei = jnp.exp(model.expected_improvement(X_meshgrid))
best = bfgs_acquisition(
    model, multi_starts=ACQUISITION_MULTISTARTS, max_iterations=MAX_ITERATIONS
)

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Expected Improvement", "Surrogate Model"),
)

# Left: Expected Improvement
fig.add_trace(
    go.Contour(
        z=ei.reshape(x_meshgrid.shape),
        x=x_grid,
        y=y_grid,
        colorbar=dict(title="EI"),
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=best[:, 0],
        y=best[:, 1],
        mode="markers",
        marker=dict(color="red", size=10),
        showlegend=False,
    ),
    row=1,
    col=1,
)

# Right: Surrogate Model mean
fig.add_trace(
    go.Contour(
        z=pred.mean.reshape(x_meshgrid.shape),
        x=x_grid,
        y=y_grid,
        colorbar=dict(title="Mean"),
    ),
    row=1,
    col=2,
)
fig.add_trace(
    go.Scatter(
        x=x0[:, 0],
        y=x0[:, 1],
        mode="markers",
        marker=dict(color="red", size=10),
        showlegend=False,
    ),
    row=1,
    col=2,
)

fig.update_layout(width=800, height=400, title="Acquisition vs Surrogate")
fig.show()

# Putting it all together

In [ ]:
models = [model]
candidates = []
for i in tqdm(range(30)):
    x = bfgs_acquisition(
        models[-1],
        multi_starts=ACQUISITION_MULTISTARTS,
        max_iterations=MAX_ITERATIONS,
    )
    y = goldstein_price(x)
    models.append(models[-1].update(x, y))

In [ ]:
fig = make_subplots(
    rows=1,
    cols=2,
    specs=[[{"type": "scene"}, {"type": "xy"}]],
    subplot_titles=("GP Prediction", "Expected Improvement"),
)
fig.add_trace(
    go.Surface(
        z=goldstein_price(X_meshgrid).reshape(x_meshgrid.shape),
        x=x_meshgrid,
        y=y_meshgrid,
        opacity=0.3,
        showscale=False,
    ),
    row=1,
    col=1,
)
x = models[-1].x[-len(models[1:]) :]
for i, (m, b) in tqdm(list(enumerate(zip(models[:-1], x)))):
    pred = m.predict(X_meshgrid)
    ei = jnp.exp(m.expected_improvement(X_meshgrid))
    fig.add_trace(
        go.Surface(
            z=pred.mean.reshape(x_meshgrid.shape),
            x=x_meshgrid,
            y=y_meshgrid,
            showscale=False,
            visible=i == 0,
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scatter3d(
            z=m.y,
            x=m.x[:, 0],
            y=m.x[:, 1],
            mode="markers",
            marker=dict(color="red", size=5),
            visible=i == 0,
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Contour(
            z=ei.reshape(x_meshgrid.shape),
            x=x_grid,
            y=y_grid,
            visible=i == 0,
        ),
        row=1,
        col=2,
    )
    fig.add_trace(
        go.Scatter(
            x=b[None, 0],
            y=b[None, 1],
            mode="markers",
            marker=dict(color="red", size=5),
            showlegend=False,
            visible=i == 0,
        ),
        row=1,
        col=2,
    )

# create slider steps
steps = []
for i in range(len(models) - 1):
    step = dict(
        method="update",
        args=[
            {
                "visible": [
                    (j == 0) or ((j - 1) // 4 == i) for j in range(1 + len(models) * 4)
                ]
            },
            {"title": f"Iteration {i}"},
        ],
    )
    steps.append(step)
sliders = [dict(active=0, steps=steps)]
fig.update_layout(sliders=sliders, title="Bayesian Optimization Progress")
fig.update_layout(scene=dict(zaxis=dict(range=[-3, 3])))
fig.update_layout(width=800, height=400)
fig.show()

In [ ]:
def run(n_init: int = 4, n_acquisitions: int = 50):
    x0 = sp.stats.qmc.LatinHypercube(d=2, optimization="random-cd").random(n=n_init)
    y0 = goldstein_price(x0)
    model = gp.GaussianProcess.fit(x0, y0, kernel=KERNEL, max_iterations=MAX_ITERATIONS, verbose=False)
    ymin = [model.y.min()]

    for i in tqdm(range(n_acquisitions)):
        x = bfgs_acquisition(
            model, max_iterations=MAX_ITERATIONS, multi_starts=ACQUISITION_MULTISTARTS
        )
        y = goldstein_price(x)
        model = model.update(x, y)
        ymin.append(model.y.min())
    return ymin


N_RUNS = 16
y = np.array([run() for _ in tqdm(range(N_RUNS))])

In [ ]:
fig = go.Figure()

for i in range(y.shape[1]):
    fig.add_trace(
        go.Box(
            y=y[:, i],
            name=str(i),
            boxmean=True,  # shows mean marker
            marker_color="blue",
            showlegend=False,
            opacity=0.5,
        )
    )
fig.add_trace(
    go.Scatter(
        x=jnp.arange(y.shape[1]),
        y=y.mean(axis=0),
        mode="lines",
        line=dict(color="green"),
        name="Minimum Found",
    )
)
fig.add_trace(
    go.Scatter(
        x=jnp.arange(y.shape[1]),
        y=goldstein_price(X_meshgrid).min() * jnp.ones_like(y[0]),
        mode="lines",
        line=dict(color="red", dash="dash"),
        name="True Global Minimum",
    )
)
fig.update_layout(
    title=f"Best Found Value Distribution per Iteration ({N_RUNS} runs)",
    xaxis_title="Iteration",
    yaxis_title="Best Found Value",
)
fig.show()